# Exploratory Data Analysis — Telco Customer Churn

This notebook explores the raw Telco Customer Churn dataset before any
feature engineering or modelling.  Goals:

1. Confirm the data loads and passes schema validation.
2. Identify missing values, duplicates, and data-type issues.
3. Understand target balance (churn rate).
4. Survey feature distributions and correlations.
5. Document preprocessing needs for the feature pipeline.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Ensure src/ is importable when running from notebooks/
SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data.load import load_telco_churn

%matplotlib inline
pd.set_option("display.max_columns", 25)

## 2. Load data

In [ ]:
df = load_telco_churn()
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe(include="all")

## 3. Missing values

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

In [ ]:
# TotalCharges contains spaces for zero-tenure rows.
# Coerce to numeric to expose the hidden NaNs.
total_charges_numeric = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"TotalCharges NaNs after coercion: {total_charges_numeric.isna().sum()}")
df[total_charges_numeric.isna()]

## 4. Duplicates

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Unique customerIDs: {df['customerID'].nunique()} / {len(df)}")

## 5. Target balance

In [ ]:
churn_counts = df["Churn"].value_counts(normalize=True)
print(churn_counts)

churn_counts.plot.bar(title="Churn distribution", rot=0)
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

## 6. Feature distributions

In [ ]:
# Numeric features
numeric_cols = ["tenure", "MonthlyCharges", "SeniorCitizen"]
df[numeric_cols].hist(bins=30, figsize=(12, 4), layout=(1, 3))
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features
cat_cols = [
    "gender", "Partner", "Dependents", "PhoneService",
    "InternetService", "Contract", "PaymentMethod",
]
for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

## 7. Correlations

In [ ]:
df_num = df.copy()
df_num["TotalCharges"] = pd.to_numeric(df_num["TotalCharges"], errors="coerce").fillna(0)
df_num["Churn_binary"] = (df_num["Churn"] == "Yes").astype(int)

corr_cols = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn_binary"]
corr = df_num[corr_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
cax = ax.matshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
fig.colorbar(cax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="left")
ax.set_yticklabels(corr.columns)
ax.set_title("Numeric correlation matrix", pad=20)
plt.tight_layout()
plt.show()

## 8. Observations and preprocessing needs

Fill in after running against the real dataset:

- **Churn rate:** ~26% (class imbalance — consider stratified splits, not oversampling yet).
- **TotalCharges:** Contains spaces for 11 zero-tenure rows — coerce to float, fill with 0.
- **Duplicates:** None expected (customerID is unique).
- **Binary columns (Yes/No):** Encode to 0/1.
- **Multi-category columns:** One-hot encode (drop first to avoid collinearity).
- **Numeric scaling:** StandardScaler for logistic regression; tree models may not need it.
- **customerID:** Drop before modelling (not predictive).

These observations feed directly into `features.preprocess.build_feature_matrix`.